# Improving Clinical Reasoning with AI : Lessons from 400 Kenyan Case Vignettes

## Business Understanding
In Kenya, frontline nurses often face high-stakes clinical decisions under intense pressure and with limited resources. These healthcare workers operate in environments where specialist support is scarce, yet their judgment can mean the difference between life and death.
The rise of large language models (LLMs) such as GPT-4, LLAMA, and GEMINI offers a potential support system for healthcare workers—either by providing second opinions or pre-screening suggestions. However, these systems must first be proven to emulate the reasoning and decision-making patterns of real, trained clinicians.
This project explores whether AI models can replicate or assist clinical decision-making in real Kenyan medical contexts.

## Problem Statement

1. Rural Kenyan healthcare workers make critical decisions with limited resources
2. 	Nurses across different counties and facility levels face complex medical situations daily
3. 	Need for AI that can match human clinical reasoning in low-resource settings

## Objectives
The goal is to train a model that can predict the clinician’s response to each complex clinical prompt, effectively mimicking the decision-making of trained healthcare professionals.


1. Build an AI model that can replicate clinical reasoning of human healthcare professionals.
2. Compare predictions from  models with outputs from existing LLMs (GPT-4, LLAMA, GEMINI).
3. Measure response similarity between predicted vs. real clinician answers using semantic metrics (e.g., BERTScore, BLEU, ROUGE).

4. Evaluate factual accuracy of responses based on DDX SNOMED codes (clinical diagnosis correctness).

5. Explore impact of nurse metadata (e.g., experience level, health facility) on model accuracy.

In [24]:
#Initialize libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import re # Import re for regular expressions
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('vader_lexicon', quiet=True)
except Exception as e:
    print(f"NLTK download failed: {e}")

In [36]:
#Load all the datasets
train_raw = pd.read_csv('data/train_raw.csv')
train_processed = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test_raw.csv')
test_processed = pd.read_csv('data/test.csv')


#For EDA and data understanding, we will use the train_raw.csv
df_original = pd.read_csv("data/train_raw.csv")
df_original.head()

,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...
2,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder)\n25458004 |...
3,ID_QOQTK,Uasin Gishu,National Referral Hospitals,12.0,I am a nurse with 12 years of experience in Pr...,Critical Care,INTERNAL MEDICINE,SUMMARY\n\n72-year-old female with inability t...,"Given ER's clinical presentation and vitals, t...",to me with this query. Based on the informatio...,This 92-year-old female patient (ER) presents ...,14760008 | Constipation (finding)\n419284004 |...
4,ID_ZFJBM,Uasin Gishu,National Referral Hospitals,16.0,I am a nurse with 16 years of experience in Ge...,Adult Health,INTERNAL MEDICINE,"A 22 year old female presents with headache, d...",The 22-year-old female patient is presenting w...,Thank you for presenting this case. Based on t...,This 22-year-old female patient presents with ...,95874006 | Carbon monoxide poisoning from fire...


## Data Understanding

In [37]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  300 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


The Data Dictionary is as follows:

<figure>
    <img src="pictures/data_dictionary.png" alt="Description" width="1500">
    <figcaption>Data Dictionary</figcaption>
</figure>

The data set has 12 columns. Some preliminary observations:
* There is missing data on the Years of Experience Column; However, the Prompt column seems to specify the nurses' years of experience and so we can attempt to extract missing data from here.
* The DDX SNOMED column can contain multiple diagnosis. For analytical purposes, we will need to explode the diagnoses
* Each DDX SNOMED entry appears to pack 3 levels of data i.e. a DDX code, its associated description and its diagnosis type. We will need to split it into these 3 levels for analytical purposes.

### Basic Data Structure Analysis

In [38]:
if train_raw is not None:


    print("\n" + "=" * 60)
    print("DATASET STRUCTURE OVERVIEW")
    print("=" * 60)

    datasets = {
        'Train Raw': train_raw,
        'Train Processed': train_processed,
        'Test Raw': test_raw,
        'Test Processed': test_processed
    }

    # Compare dataset shapes
    print("\n Dataset Dimensions:")
    for name, df in datasets.items():
        print(f"{name:15}: {df.shape[0]:3d} rows × {df.shape[1]:2d} columns")

    # Check columns in each dataset
    print("\n Column Comparison:")
    for name, df in datasets.items():
        print(f"\n{name} Columns ({len(df.columns)}):")
        print(df.columns.tolist())


DATASET STRUCTURE OVERVIEW

 Dataset Dimensions:
Train Raw      : 400 rows × 12 columns
Train Processed: 400 rows × 12 columns
Test Raw       : 100 rows ×  7 columns
Test Processed : 100 rows ×  7 columns

 Column Comparison:

Train Raw Columns (12):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI', 'DDX SNOMED']

Train Processed Columns (12):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI', 'DDX SNOMED']

Test Raw Columns (7):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel']

Test Processed Columns (7):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel']


### Tableau  EDA

Exploratory Data Analysis was done using Tableau tools

<figure>
    <img src="pictures/Medical_Categories_Distribution.png" alt="Description" width="1500">
    <figcaption>Medical Categories Distribution</figcaption>
</figure>

Internal Medicine and Surgery comprise 50% of the cases in these hospitals.

<figure>
    <img src="pictures/Data_Distribution_By_County.png" alt="Description" width="1500">
    <figcaption>MData Distribution By County</figcaption>
</figure>

Majority of the data was collected from medical facilities in Uasin Gishu county

<figure>
    <img src="pictures/Years_of_Experience_Statistical_Summary_by_County.png" alt="Description" width="700">
    <figcaption>Years of Experience Statistical Summary by County</figcaption>
</figure>

Uasin Gishu county generally has more experience nurses

<figure>
    <img src="pictures/Years_of_Experience_Statistical_Summary_by_County_by_facility_type.png" alt="Description" width="1500">
    <figcaption>Years of Experience Statistical Summary by County and Facility Type</figcaption>
</figure>

Uasin Gishu Health Centres are the Medical Facility type with the highest Nurse Experience

<figure>
    <img src="pictures/Nurses_Average_Years_of_Experience_Per_County_Per_Facility_Type.png" alt="Description" width="1500">
    <figcaption>Nurses' Average Years of Experience Per County Per Facility Type</figcaption>
</figure>

The regional average number of years of experience by medical facility type can be used to impute missing Years of Experience data

<figure>
    <img src="pictures/Nurses_Median_Years_of_Experience_Per_County_Per_Facility_Type.png" alt="Description" width="1500">
    <figcaption>Nurses' Median Years of Experience Per County Per Facility Type</figcaption>
</figure>

The regional median number of years of experience by medical facility type can be used to impute missing Years of Experience data

<figure>
    <img src="pictures/Data_Distribution_by_Nursing_Competency.png" alt="Description" width="1500">
    <figcaption>Data Distribution by Nursing Competency</figcaption>
</figure>

Adult Health and General Emergency cases comprise approximately 50% of the cases in these regions

A few summaries from the data are shown below

In [39]:
if train_raw is not None:
    print("\n" + "=" * 60)
    print("CATEGORICAL FEATURES ANALYSIS")
    print("=" * 60)

    categorical_cols = ['County', 'Health level', 'Nursing Competency', 'Clinical Panel']

    for col in categorical_cols:
        if col in train_raw.columns:
            print(f"\n {col}:")
            value_counts = train_raw[col].value_counts()
            print(f"  Total unique values: {len(value_counts)}")
            print("  Top 5 categories:")
            for val, count in value_counts.head().items():
                percentage = (count / len(train_raw)) * 100
                print(f"    {val}: {count} ({percentage:.1f}%)")


CATEGORICAL FEATURES ANALYSIS

 County:
  Total unique values: 5
  Top 5 categories:
    Uasin Gishu: 247 (61.8%)
    Kakamega: 83 (20.8%)
    Kiambu: 60 (15.0%)
    Elgeiyo Marakwet: 6 (1.5%)
    Bungoma: 4 (1.0%)

 Health level:
  Total unique values: 8
  Top 5 categories:
    Sub-county Hospitals and Nursing Homes: 131 (32.8%)
    National Referral Hospitals: 125 (31.2%)
    Health centres: 69 (17.2%)
    Dispensaries and Private Clinics: 54 (13.5%)
    County Hospitals: 9 (2.2%)

 Nursing Competency:
  Total unique values: 21
  Top 5 categories:
    Adult Health: 123 (30.8%)
    General Emergency: 66 (16.5%)
    Child Health: 56 (14.0%)
    Maternal and Child Health: 50 (12.5%)
    Sexual And Reproductive Health: 25 (6.2%)

 Clinical Panel:
  Total unique values: 14
  Top 5 categories:
    INTERNAL MEDICINE: 133 (33.2%)
    SURGERY: 91 (22.8%)
    PAEDIATRICS: 78 (19.5%)
    OBSTETRICS AND GYNAECOLOGY: 68 (17.0%)
    CRITICAL CARE: 18 (4.5%)


### Text Data Structure Analysis

In [40]:
if train_raw is not None:

    print("\n" + "=" * 60)
    print("TEXT DATA ANALYSIS")
    print("=" * 60)

    text_columns = ['Prompt', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI']

    for col in text_columns:
        if col in train_raw.columns:
            print(f"\n {col} Text Statistics:")

            # Calculate text lengths
            text_lengths = train_raw[col].astype(str).str.len()
            word_counts = train_raw[col].astype(str).str.split().str.len()

            print(f"  Character length - Mean: {text_lengths.mean():.0f}, Median: {text_lengths.median():.0f}")
            print(f"  Word count - Mean: {word_counts.mean():.0f}, Median: {word_counts.median():.0f}")
            print(f"  Min length: {text_lengths.min()}, Max length: {text_lengths.max()}")


TEXT DATA ANALYSIS

 Prompt Text Statistics:
  Character length - Mean: 556, Median: 542
  Word count - Mean: 96, Median: 92
  Min length: 229, Max length: 1403

 Clinician Text Statistics:
  Character length - Mean: 722, Median: 676
  Word count - Mean: 108, Median: 102
  Min length: 155, Max length: 2143

 GPT4.0 Text Statistics:
  Character length - Mean: 5419, Median: 5384
  Word count - Mean: 775, Median: 780
  Min length: 1278, Max length: 17451

 LLAMA Text Statistics:
  Character length - Mean: 2324, Median: 2308
  Word count - Mean: 335, Median: 331
  Min length: 1060, Max length: 3501

 GEMINI Text Statistics:
  Character length - Mean: 3857, Median: 3970
  Word count - Mean: 535, Median: 554
  Min length: 1071, Max length: 5576


###   Clinician vs NLP Responses Text Summary

| **Model**         | **Avg Characters** | **Avg Words** | **Style Implication**                 |
|------------------|-------------------|---------------|---------------------------------------|
| **Human Clinician** | 722               | 108           | Concise, focused                      |
| **LLAMA**         | 2,324             | 335           | ~3× longer than human                 |
| **GEMINI**        | 3,857             | 535           | ~5× longer than human                 |
| **GPT-4**         | 5,419             | 775           | ~7× longer than human                 |

---

**Critical Insight**:Human clinicians are significantly more concise than AI models.
To improve clinical utility, the model needs to **learn to be brief and focused**—not overly verbose.

### Sample Data Inspection

In [41]:
if train_raw is not None:

    print("\n" + "=" * 60)
    print("SAMPLE DATA INSPECTION")
    print("=" * 60)

    print("\n First Training Sample:")
    sample_idx = 50
    sample = train_raw.iloc[sample_idx]

    key_fields = ['Master_Index', 'County', 'Health level', 'Nursing Competency', 'Clinical Panel']
    for field in key_fields:
        if field in sample.index:
            print(f"  {field}: {sample[field]}")

    print(f"\n Prompt (truncated):")
    prompt_text = str(sample.get('Prompt', 'N/A'))
    print(f"  {prompt_text[:300]}{'...' if len(prompt_text) > 300 else ''}")

    print(f"\n Clinician Response (truncated):")
    clinician_text = str(sample.get('Clinician', 'N/A'))
    print(f"  {clinician_text[:300]}{'...' if len(clinician_text) > 300 else ''}")


SAMPLE DATA INSPECTION

 First Training Sample:
  Master_Index: ID_ICCWP
  County: Kiambu
  Health level: Sub-county Hospitals and Nursing Homes
  Nursing Competency: General Emergency
  Clinical Panel: CRITICAL CARE

 Prompt (truncated):
  I am a nurse with 12 years of experience in General nursing working in a Sub-county Hospitals and Nursing Homes in Kiambu county in Kenya. A female patient is brought to the hospital with complaint of shortness of breath upon exhaustion for three days. On examination, temperature is at 39.4, blood p...

 Clinician Response (truncated):
  SUMMARY
Female, SOB 3/7, T 39.4, BP 122/80, SpO2 80. Oxygen is empty.

How do I manage this patient?
Assess for life-threatening conditions, administer antipyretics and monitor vitals, start fluid therapy.

Perform the following laboratory tests:
 full hemogram, liver and renal functions, ESR and pr...


### Data Cleaning

#### Handle Missing Values on the Years of Experience Column
We will attempt to extract Years of Experience from the Nurses' prompts if possible

In [ ]:
#Create a function that extracts level of experience from the prompt, 
def extract_experience(prompt):
    match = re.search(r"(\d+)[ ]*years of experience", prompt)
    return int(match.group(1)) if match else None

df_original['Extracted Experience'] = df_original['Prompt'].apply(extract_experience) #Extract the experience level from the nurses prompt
df_original['Experience Match'] = df_original['Extracted Experience'] == df_original['Years of Experience'] #check if the extracted experience level matches the years of experience level column and score True or False
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Master_Index          400 non-null    object 
 1   County                400 non-null    object 
 2   Health level          400 non-null    object 
 3   Years of Experience   300 non-null    float64
 4   Prompt                400 non-null    object 
 5   Nursing Competency    400 non-null    object 
 6   Clinical Panel        400 non-null    object 
 7   Clinician             400 non-null    object 
 8   GPT4.0                400 non-null    object 
 9   LLAMA                 400 non-null    object 
 10  GEMINI                400 non-null    object 
 11  DDX SNOMED            399 non-null    object 
 12  Extracted Experience  300 non-null    float64
 13  Experience Match      400 non-null    bool   
dtypes: bool(1), float64(2), object(11)
memory usage: 41.1+ KB


Even after attempting to years of experience from the prompt, there is still missing data. A manual inspection of some rows with this missing data showed that these nurses did not specify their experience level in their prompt. As experience level may be crucial in the model, we decided to impute by assigning it the mean of nurses in that county and that facility type.

We can then drop these 2 new columns as we do not need them any further

In [44]:
df_original = df_original.drop(['Extracted Experience', 'Experience Match'], axis=1)
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  300 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


Impute the missing Years of Experience using the avaerage for that county and medical facility type

In [46]:
#Impute missing Years of Experience with mean based on that county and that facility type
df_original['Years of Experience'] = df_original.apply(
    lambda row: df_original[(df_original['County'] == row['County']) & (df_original['Health level'] == row['Health level'])]['Years of Experience'].mean()
    if pd.isna(row['Years of Experience']) else row['Years of Experience'], axis=1
)
df_cleaned = df_original.copy()
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  400 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


#### Exploding DDX SNOMED Values 
We will attempt to explode the Data to seperate multiple diagnoses into individual rows

In [51]:
df_exploded = df_cleaned.copy()
df_exploded["DDX SNOMED Split"] = df_exploded["DDX SNOMED"].str.split("\n")
df_exploded = df_exploded.explode("DDX SNOMED Split").reset_index(drop=True)
df_exploded.drop(columns=["DDX SNOMED"], inplace=True)
df_exploded.head()


,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED Split
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...
2,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",700055009 | Sepsis with cutaneous manifestatio...
3,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder)
4,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",25458004 | Acute gastritis (disorder)


In [52]:
df_exploded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         1202 non-null   object 
 1   County               1202 non-null   object 
 2   Health level         1202 non-null   object 
 3   Years of Experience  1202 non-null   float64
 4   Prompt               1202 non-null   object 
 5   Nursing Competency   1202 non-null   object 
 6   Clinical Panel       1202 non-null   object 
 7   Clinician            1202 non-null   object 
 8   GPT4.0               1202 non-null   object 
 9   LLAMA                1202 non-null   object 
 10  GEMINI               1202 non-null   object 
 11  DDX SNOMED Split     1201 non-null   object 
dtypes: float64(1), object(11)
memory usage: 112.8+ KB


The data has now been exploded such that each diagnosis appears on its own row. Next, we need to engineer the 3 layers of packed information from the DDX SNOMED Split column into a DDX code, its description and DDX Type

In [53]:
# Function to extract Code, Description, and Type from the DDX SNOMED Column
def extract_snomed_details(snomed_split_str):
    if pd.isna(snomed_split_str):
        return pd.NA, pd.NA, pd.NA

    # Using regex to capture the code, description, and the part in parentheses
    match = re.match(r'(\d+)\s*\|\s*(.*?)\s*\((.*?)\)', snomed_split_str)

    if match:
        code = match.group(1)
        description = match.group(2).strip()
        snomed_type = match.group(3).strip()
        return code, description, snomed_type
    else:
        # Handle cases that don't match the expected format
        return pd.NA, pd.NA, pd.NA

# Apply the function to the 'DDX SNOMED Split' column
df_exploded[['DDX Code', 'DDX Description', 'DDX Type']] = df_exploded['DDX SNOMED Split'].apply(
    lambda x: pd.Series(extract_snomed_details(x))
)

df_exploded.head()

,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED Split,DDX Code,DDX Description,DDX Type
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...,288514009,Burn involving 5 percent of body surface,disorder
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...,420270002,Ketoacidosis due to type 1 diabetes mellitus,disorder
2,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",700055009 | Sepsis with cutaneous manifestatio...,700055009,Sepsis with cutaneous manifestations,disorder
3,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder),13200003,Peptic ulcer,disorder
4,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",25458004 | Acute gastritis (disorder),25458004,Acute gastritis,disorder


With this clean exploded data, we can now analyze the data further e.g. frequency of certain diagnoses, distribution of diagnoses per county or per facility type (or both) etc.
Later, we could attempt extracting sentiments from the prompts that match with the descriptions from the DDX SNOMED.

In [54]:
df_exploded.to_csv("data/df_exploded.csv", index=False)